In [2]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.data_ingestion import ingest_data

df = ingest_data()

df_en = df[df["language"] == "en"].copy()

df_en["subject"] = df_en["subject"].fillna("")
df_en["body"] = df_en["body"].fillna("")

df_en["text"] = (
    df_en["subject"] + " " + df_en["body"]
).str.strip()

print("English tickets:", len(df_en))

Extracting: multilingual-customer-support-tickets.zip
Loading: aa_dataset-tickets-multi-lang-5-2-50-version.csv

Data ingestion completed.
File: aa_dataset-tickets-multi-lang-5-2-50-version.csv
Shape: (28587, 16)
English tickets: 16338


In [3]:
from sklearn.model_selection import train_test_split

X = df_en["text"]
y = df_en["queue"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 13070
Testing samples: 3268


In [4]:
import joblib

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"

queue_tfidf = joblib.load(
    MODELS_DIR / "queue_tfidf_vectorizer.pkl"
)

queue_model = joblib.load(
    MODELS_DIR / "queue_classifier.pkl"
)

print("Queue model loaded successfully.")

Queue model loaded successfully.


In [5]:
X_test_tfidf = queue_tfidf.transform(X_test)

y_pred = queue_model.predict(X_test_tfidf)

print("Predictions generated.")
print("Number of predictions:", len(y_pred))

Predictions generated.
Number of predictions: 3268


In [6]:
error_df = pd.DataFrame({
    "text": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred
})

error_df["correct"] = (
    error_df["actual"] == error_df["predicted"]
)

print("Total test tickets:", len(error_df))
print("Correct predictions:", error_df["correct"].sum())
print("Incorrect predictions:", (~error_df["correct"]).sum())

Total test tickets: 3268
Correct predictions: 2183
Incorrect predictions: 1085


In [7]:
confusion_pairs = (
    error_df[error_df["correct"] == False]
    .groupby(["actual", "predicted"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confusion_pairs.head(15)

,actual,predicted,count
43,Product Support,Technical Support,110
67,Technical Support,IT Support,88
68,Technical Support,Product Support,86
34,IT Support,Technical Support,77
64,Technical Support,Customer Service,64
15,Customer Service,Technical Support,63
36,Product Support,Customer Service,51
11,Customer Service,Product Support,41
39,Product Support,IT Support,40
30,IT Support,Product Support,32


In [8]:
mistakes = error_df[
    error_df["correct"] == False
].copy()

mistakes[
    ["actual", "predicted", "text"]
].head(10)

,actual,predicted,text
0,Technical Support,Product Support,Issue Encountered Assistance Needed
1,Technical Support,Product Support,Hospital Systems Data Breach Notification In r...
5,General Inquiry,Product Support,Support for Investment Optimization Issues Cus...
7,Product Support,IT Support,Concern Regarding Security of Health Data Rece...
10,Customer Service,Billing and Payments,Details on Digital Strategies for Brand Growth...
14,Technical Support,Customer Service,Request for PostgreSQL Compatibility Details S...
15,Customer Service,Product Support,Looking for assistance with a problem regardin...
17,Product Support,IT Support,Urgent Report of Security Breach Dear Customer...
19,Customer Service,Returns and Exchanges,Query for Data Analytics Assistance Enthusiast...
21,IT Support,Technical Support,Software Malfunction The digital marketing cam...
